In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from sklearn.covariance import MinCovDet
import os
import time
from sklearn.utils.extmath import fast_logdet

In [2]:
import numpy as np
import pandas as pd
import hdbscan
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd 
import os

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA  # <-- Missing import
import matplotlib.pyplot as plt
#data load
raw_counts = pd.read_csv("for_ml.csv", index_col=0)

In [3]:
raw_counts.head()

,Treatment,Treatment.1,Control,Control,Treatment.2,Treatment.3,Control.1,Control.2
GeneID,,,,,,,,
LOC101497325,18,15,29,11,9,18,29,20
LOC101488339,19,39,42,21,6,36,41,21
LOC101488545,40,33,33,36,25,43,49,38
LOC101497647,113,93,227,50,52,62,253,90
LOC101489113,1323,1417,1627,1011,1349,1450,1562,1708


In [4]:
raw_counts.columns = raw_counts.columns.str.replace(r'^(Control|Treatment).*', r'\1', regex=True)
print("\nCleaned columns:")
print(raw_counts.columns)


Cleaned columns:
Index(['Treatment', 'Treatment', 'Control', 'Control', 'Treatment',
       'Treatment', 'Control', 'Control'],
      dtype='object')


In [5]:
filtered_df = raw_counts

BELOW is for 5k

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
import hdbscan
from scipy import linalg

def fast_logdet(matrix):
    """Compute log determinant efficiently"""
    return 2 * np.sum(np.log(np.diag(linalg.cholesky(matrix))))

# Create output directory
output_dir = "final_results/distance_metric_comparison_again_for5k"
os.makedirs(output_dir, exist_ok=True)

# 1. Data preparation
normalized_data = np.log2(filtered_df + 1)  # Log2 transform with pseudocount

# 2. Select top variable genes
gene_variability = normalized_data.std(axis=1)
top_genes = gene_variability.sort_values(ascending=False).head(5000).index
top_variable_data = normalized_data.loc[top_genes]

print(f"Top variable genes shape: {top_variable_data.shape}")

# 3. Z-score normalization
# MISTAKE WAS HERE: Using top_genes_idx instead of top_genes for the index
# Original (wrong): index=top_genes_idx
# Corrected: index=top_genes
top_variable_data = pd.DataFrame(
    StandardScaler().fit_transform(top_variable_data),
    index=top_genes,  # CORRECTED: Changed from top_genes_idx to top_genes
    columns=top_variable_data.columns
)

# ----------------------------
class RobustMahalanobisHDBSCAN:
    def __init__(self, min_cluster_size=5, min_samples=None):
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        
    def _robust_mahalanobis(self, X):
        """Calculate pairwise Mahalanobis distances with regularization"""
        try:
            # Robust covariance estimation with increased support_fraction
            cov_estimator = MinCovDet(support_fraction=0.9).fit(X)
            cov = cov_estimator.covariance_
            
            # Regularization to ensure invertibility
            reg = 1e-6 * np.trace(cov)/cov.shape[0] * np.eye(cov.shape[0])
            cov_reg = cov + reg
            
            # Check condition number
            if fast_logdet(cov_reg) < -50:  # If nearly singular
                raise ValueError("Covariance matrix is nearly singular")
                
            cov_inv = np.linalg.inv(cov_reg)
            
            # Calculate distances using matrix operations
            diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
            dists = np.sqrt(np.einsum('...i,ij,...j->...', diff, cov_inv, diff))
            np.fill_diagonal(dists, 0)  # Ensure diagonal is exactly 0
            return dists
            
        except Exception as e:
            print(f"Mahalanobis calculation failed: {str(e)}")
            return None

    def fit(self, X):
        dist_matrix = self._robust_mahalanobis(X)
        if dist_matrix is None:
            # Fallback to Euclidean if Mahalanobis fails
            print("Using Euclidean as fallback")
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples
            )
        else:
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='precomputed'
            )
            
        self.labels_ = self.clusterer.fit_predict(dist_matrix if dist_matrix is not None else X)
        return self

# ----------------------------
# Metric Comparison
# ----------------------------
metrics = [
    ('euclidean', hdbscan.HDBSCAN(metric='euclidean')),
    ('manhattan', hdbscan.HDBSCAN(metric='manhattan')),
    ('correlation', hdbscan.HDBSCAN(metric='correlation')),
    ('mahalanobis', RobustMahalanobisHDBSCAN())
]

results = []
for name, clusterer in metrics:
    print(f"\n{'='*40}\nClustering with: {name.upper()}\n{'='*40}")
    
    try:
        start_time = time.time()
        
        if name == 'mahalanobis':
            print("Computing Mahalanobis distances...")
            clusterer.fit(top_variable_data.values)
            clusters = clusterer.labels_
        else:
            clusters = clusterer.fit_predict(top_variable_data)
            
        # Analysis
        unique_clusters = np.unique(clusters)
        n_clusters = len(unique_clusters) - 1  # Exclude noise (-1)
        noise_ratio = np.mean(clusters == -1)
        cluster_sizes = [np.sum(clusters == c) for c in unique_clusters if c != -1]
        
        results.append({
            'metric': name,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'clusters': clusters,
            'cluster_sizes': cluster_sizes,
            'time': time.time() - start_time
        })
        
        print(f"Number of clusters: {n_clusters}")
        print(f"Noise ratio: {noise_ratio:.1%}")
        print(f"Cluster sizes: {sorted(cluster_sizes, reverse=True)}")
        print(f"Time elapsed: {results[-1]['time']:.2f} seconds")
        
    except Exception as e:
        print(f"Clustering failed: {str(e)}")
        results.append({
            'metric': name,
            'error': str(e),
            'clusters': None
        })

# ----------------------------
# Visualization
# ----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

pca = PCA(n_components=2).fit_transform(top_variable_data)

for i, result in enumerate(results):
    ax = axes[i]
    
    if result['clusters'] is None:
        ax.text(0.5, 0.5, f"Failed:\n{result.get('error','')}", 
                ha='center', va='center', fontsize=12)
        ax.set_title(f"{result['metric'].upper()} - FAILED", fontsize=14)
        continue
        
    sc = ax.scatter(
        pca[:, 0], pca[:, 1],
        c=result['clusters'],
        cmap='tab20',
        alpha=0.7,
        s=15
    )
    
    title = (
        f"{result['metric'].upper()}\n"
        f"Clusters: {result['n_clusters']} | "
        f"Noise: {result['noise_ratio']:.1%}\n"
        f"Time: {result['time']:.2f}s"
    )
    ax.set_title(title, fontsize=12)
    
    # Add colorbar for cluster IDs
    plt.colorbar(sc, ax=ax, label='Cluster ID')

plt.tight_layout()
plt.savefig(f"{output_dir}/metric_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Cluster Size Distribution Plot
# ----------------------------
plt.figure(figsize=(12, 6))
for result in results:
    if result['clusters'] is not None and result['cluster_sizes']:
        sizes = result['cluster_sizes']
        plt.plot(sorted(sizes, reverse=True), 'o-', label=result['metric'].upper())

plt.xlabel("Cluster Rank", fontsize=12)
plt.ylabel("Number of Genes", fontsize=12)
plt.title("Cluster Size Distribution by Distance Metric", fontsize=14)
plt.yscale('log')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.savefig(f"{output_dir}/cluster_size_distribution.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Save Results
# ----------------------------
for result in results:
    if result['clusters'] is not None:
        pd.DataFrame({
            'gene': top_variable_data.index,
            'cluster': result['clusters']
        }).to_csv(f"{output_dir}/clusters_{result['metric']}.csv", index=False)

# Save summary statistics
summary_data = []
for r in results:
    summary_data.append({
        'metric': r['metric'],
        'n_clusters': r.get('n_clusters', np.nan),
        'noise_ratio': r.get('noise_ratio', np.nan),
        'largest_cluster': max(r['cluster_sizes']) if 'cluster_sizes' in r and r['cluster_sizes'] else np.nan,
        'time_seconds': r.get('time', np.nan),
        'status': 'SUCCESS' if r['clusters'] is not None else 'FAILED'
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(f"{output_dir}/summary_stats.csv", index=False)

print(f"\nAnalysis complete. Results saved to: {output_dir}")
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))

Top variable genes shape: (5000, 8)

Clustering with: EUCLIDEAN


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 40
Noise ratio: 70.7%
Cluster sizes: [np.int64(1135), np.int64(19), np.int64(16), np.int64(16), np.int64(16), np.int64(14), np.int64(14), np.int64(12), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(6), np.int64(6), np.int64(6), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5)]
Time elapsed: 0.21 seconds

Clustering with: MANHATTAN


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 34
Noise ratio: 84.2%
Cluster sizes: [np.int64(346), np.int64(55), np.int64(35), np.int64(35), np.int64(31), np.int64(28), np.int64(22), np.int64(19), np.int64(19), np.int64(17), np.int64(14), np.int64(14), np.int64(14), np.int64(12), np.int64(11), np.int64(10), np.int64(9), np.int64(8), np.int64(8), np.int64(7), np.int64(7), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5)]
Time elapsed: 0.24 seconds

Clustering with: CORRELATION


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of clusters: 48
Noise ratio: 85.5%
Cluster sizes: [np.int64(192), np.int64(45), np.int64(43), np.int64(38), np.int64(23), np.int64(23), np.int64(21), np.int64(19), np.int64(18), np.int64(15), np.int64(14), np.int64(13), np.int64(12), np.int64(11), np.int64(11), np.int64(11), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5)]
Time elapsed: 0.43 seconds

Clustering with: MAHALANOBIS
Computing Mahalanobis distances...
Number of clusters: 42
Noise ratio: 83.8%
Cluster sizes: [np.int64(441), np.int64(23), np.int64(21), np.int64(19), np.int64(18), np.int64(17), np.int64(16), np.int64(15), np.int64(12), np.int64(12), np.int6